In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, LSTM, GRU, Dense, Dropout
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import time
from tensorflow.keras.callbacks import Callback

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', })

class TimingCallback(Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_time_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1} time: {time.time() - self.epoch_time_start:.2f}s")

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
data_dir = "UCI_HAR_Dataset"
if not os.path.exists("UCI HAR Dataset"):
    print("Downloading UCI HAR Dataset...")
    urllib.request.urlretrieve(url, "har.zip")
    with zipfile.ZipFile("har.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extracted.")

def load_har_data(subset='train'):
    path = f"UCI HAR Dataset/{subset}/Inertial Signals/"
    signals = [
        "body_acc_x_", "body_acc_y_", "body_acc_z_",
        "body_gyro_x_", "body_gyro_y_", "body_gyro_z_",
        "total_acc_x_", "total_acc_y_", "total_acc_z_"
    ]
    loaded = []
    for sig in signals:
        loaded.append(np.loadtxt(f"{path}{sig}{subset}.txt"))
    X = np.dstack(loaded)
    y = np.loadtxt(f"UCI HAR Dataset/{subset}/y_{subset}.txt") - 1
    return X, y

print("Loading data...")
X_train_full, y_train_full = load_har_data('train')
X_test_full, y_test_full = load_har_data('test')

np.random.seed(42)
tf.random.set_seed(42)
idx = np.arange(len(X_train_full))
np.random.shuffle(idx)
X_train_full = X_train_full[idx[:3000]]
y_train_full = y_train_full[idx[:3000]]

X_train = X_train_full[:2100]
y_train = y_train_full[:2100]
X_val = X_train_full[2100:2550]
y_val = y_train_full[2100:2550]
X_test_sub = X_test_full
y_test_sub = y_test_full

mean = np.mean(X_train, axis=(0, 1))
std = np.std(X_train, axis=(0, 1))
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test_sub = (X_test_sub - mean) / std

print("Input tensor shape:")
print(f"Training   : {X_train.shape}")
print(f"Validation : {X_val.shape}")
print(f"Testing    : {X_test_sub.shape}")

plt.figure(figsize=(10, 6))
classes_to_plot = [0, 3, 5]
class_names = ["Walking", "Walking Upstairs", "Walking Downstairs", "Sitting", "Standing", "Laying"]
for i, c in enumerate(classes_to_plot):
    idx = np.where(y_train == c)[0][0]
    plt.subplot(3, 1, i+1)
    plt.plot(X_train[idx, :, 0], label='body_acc_x', linewidth=2)
    plt.plot(X_train[idx, :, 3], label='body_gyro_x', linewidth=2)
    plt.plot(X_train[idx, :, 6], label='total_acc_x', linewidth=2)
    plt.title(f"Activity: {class_names[c]}", fontweight='bold')
    plt.xlabel("Time step", fontweight='bold')
    plt.ylabel("Sensor value", fontweight='bold')
    plt.legend(prop={'weight':'bold'})
plt.tight_layout()
plt.savefig("Plot1_SensorSignals.eps", format='eps', dpi=600)
plt.close()

x1, x2, x3 = 0.5, 0.7, 0.2
h0, Wx, Wh, b = 0.0, 0.5, 0.8, 0.1
h1 = np.tanh(Wx*x1 + Wh*h0 + b)
h2 = np.tanh(Wx*x2 + Wh*h1 + b)
h3 = np.tanh(Wx*x3 + Wh*h2 + b)
print(f"Numerical Exercise (Manual): h1={h1:.4f}, h2={h2:.4f}, h3={h3:.4f}")

def build_model(layer_type, units=32):
    model = Sequential()
    model.add(Input(shape=(128, 9)))
    if layer_type == 'RNN':
        model.add(SimpleRNN(units))
    elif layer_type == 'LSTM':
        model.add(LSTM(units))
    elif layer_type == 'GRU':
        model.add(GRU(units))
    model.add(Dropout(0.2))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(6, activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

models = {'RNN': build_model('RNN'), 'LSTM': build_model('LSTM'), 'GRU': build_model('GRU')}
histories = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    start_time = time.time()
    histories[name] = model.fit(X_train, y_train, validation_data=(X_val, y_val), 
                                batch_size=32, epochs=30, verbose=1, callbacks=[TimingCallback()])
    train_time = time.time() - start_time
    
    y_pred = np.argmax(model.predict(X_test_sub, verbose=0), axis=-1)
    acc = accuracy_score(y_test_sub, y_pred)
    prec = precision_score(y_test_sub, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test_sub, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test_sub, y_pred, average='macro', zero_division=0)
    params = model.count_params()
    cm = confusion_matrix(y_test_sub, y_pred)
    
    results[name] = {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'time': train_time, 'params': params, 'cm': cm}

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix - {name}", fontweight='bold')
    plt.colorbar()
    tick_marks = np.arange(6)
    plt.xticks(tick_marks, class_names, rotation=45, fontweight='bold')
    plt.yticks(tick_marks, class_names, fontweight='bold')
    plt.ylabel('True label', fontweight='bold')
    plt.xlabel('Predicted label', fontweight='bold')
    for i in range(6):
        for j in range(6):
            plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center", color="white" if cm[i, j] > cm.max() / 2. else "black", fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"Plot4_CM_{name}.eps", format='eps', dpi=600)
    plt.close()

plt.figure(figsize=(15, 5))
for i, name in enumerate(['RNN', 'LSTM', 'GRU']):
    plt.subplot(1, 3, i+1)
    plt.plot(histories[name].history['loss'], label='Train Loss', linewidth=2)
    plt.plot(histories[name].history['val_loss'], label='Val Loss', linewidth=2)
    plt.title(f"{name} Loss", fontweight='bold')
    plt.xlabel("Epoch", fontweight='bold')
    plt.ylabel("Loss", fontweight='bold')
    plt.legend(prop={'weight':'bold'})
plt.tight_layout()
plt.savefig("Plot2_Loss.eps", format='eps', dpi=600)
plt.close()

plt.figure(figsize=(15, 5))
for i, name in enumerate(['RNN', 'LSTM', 'GRU']):
    plt.subplot(1, 3, i+1)
    plt.plot(histories[name].history['accuracy'], label='Train Acc', linewidth=2)
    plt.plot(histories[name].history['val_accuracy'], label='Val Acc', linewidth=2)
    plt.title(f"{name} Accuracy", fontweight='bold')
    plt.xlabel("Epoch", fontweight='bold')
    plt.ylabel("Accuracy (%)", fontweight='bold')
    plt.legend(prop={'weight':'bold'})
plt.tight_layout()
plt.savefig("Plot3_Accuracy.eps", format='eps', dpi=600)
plt.close()

labels = ['Accuracy', 'Macro F1', 'Norm Time (s/10)']
rnn_metrics = [results['RNN']['acc'], results['RNN']['f1'], results['RNN']['time']/10]
lstm_metrics = [results['LSTM']['acc'], results['LSTM']['f1'], results['LSTM']['time']/10]
gru_metrics = [results['GRU']['acc'], results['GRU']['f1'], results['GRU']['time']/10]

x = np.arange(len(labels))
width = 0.25
plt.figure(figsize=(10, 6))
plt.bar(x - width, rnn_metrics, width, label='RNN')
plt.bar(x, lstm_metrics, width, label='LSTM')
plt.bar(x + width, gru_metrics, width, label='GRU')
plt.ylabel('Score / Value', fontweight='bold')
plt.title('Model Performance Comparison', fontweight='bold')
plt.xticks(x, labels, fontweight='bold')
plt.legend(prop={'weight':'bold'})
plt.tight_layout()
plt.savefig("Plot5_Comparison.eps", format='eps', dpi=600)
plt.close()

print("\n--- Performance ---")
for name in ['RNN', 'LSTM', 'GRU']:
    res = results[name]
    print(f"{name}: Acc: {res['acc']:.4f}, Prec: {res['prec']:.4f}, Rec: {res['rec']:.4f}, F1: {res['f1']:.4f}, Params: {res['params']}, Time: {res['time']:.2f}s")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, LSTM, GRU, Dense, Dropout
from sklearn.metrics import f1_score
import time

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', })

def load_har_data(subset='train'):
    path = f"UCI HAR Dataset/{subset}/Inertial Signals/"
    signals = [
        "body_acc_x_", "body_acc_y_", "body_acc_z_",
        "body_gyro_x_", "body_gyro_y_", "body_gyro_z_",
        "total_acc_x_", "total_acc_y_", "total_acc_z_"
    ]
    loaded = []
    for sig in signals:
        loaded.append(np.loadtxt(f"{path}{sig}{subset}.txt"))
    X = np.dstack(loaded)
    y = np.loadtxt(f"UCI HAR Dataset/{subset}/y_{subset}.txt") - 1
    return X, y

print("Loading data...")
X_train_full, y_train_full = load_har_data('train')
X_test_full, y_test_full = load_har_data('test')

np.random.seed(42)
tf.random.set_seed(42)
idx = np.arange(len(X_train_full))
np.random.shuffle(idx)
X_train_full = X_train_full[idx[:3000]]
y_train_full = y_train_full[idx[:3000]]

X_train = X_train_full[:2100]
y_train = y_train_full[:2100]
X_val = X_train_full[2100:2550]
y_val = y_train_full[2100:2550]

X_test_sub = X_test_full
y_test_sub = y_test_full

def build_model(layer_type, seq_len, units=32):
    model = Sequential()
    model.add(Input(shape=(seq_len, 9)))
    if layer_type == 'RNN':
        model.add(SimpleRNN(units))
    elif layer_type == 'LSTM':
        model.add(LSTM(units))
    elif layer_type == 'GRU':
        model.add(GRU(units))
    model.add(Dropout(0.2))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(6, activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

seq_lengths = [32, 64, 128]
models_to_test = ['RNN', 'LSTM', 'GRU']
f1_results = {m: [] for m in models_to_test}

print("\n--- Testing Sequence Lengths ---")
for seq_len in seq_lengths:
    X_tr = X_train[:, -seq_len:, :]
    X_ts = X_test_sub[:, -seq_len:, :]
    
    mean = np.mean(X_tr, axis=(0, 1))
    std = np.std(X_tr, axis=(0, 1))
    X_tr = (X_tr - mean) / std
    X_ts = (X_ts - mean) / std
    
    for name in models_to_test:
        model = build_model(name, seq_len)
        print(f"Training {name} with T={seq_len}...")
        model.fit(X_tr, y_train, batch_size=32, epochs=15, verbose=0)
        y_pred = np.argmax(model.predict(X_ts, verbose=0), axis=-1)
        f1 = f1_score(y_test_sub, y_pred, average='macro', zero_division=0)
        f1_results[name].append(f1)
        print(f"{name} (T={seq_len}) F1: {f1:.4f}")

plt.figure(figsize=(8, 6))
for name in models_to_test:
    plt.plot(seq_lengths, f1_results[name], marker='o', label=name, linewidth=2)
plt.title('Sequence Length vs Test F1-score', fontweight='bold')
plt.xlabel('Sequence Length', fontweight='bold')
plt.ylabel('Macro F1-score', fontweight='bold')
plt.xticks(seq_lengths, fontweight='bold')
plt.legend(prop={'weight':'bold'})
plt.grid(True)
plt.tight_layout()
plt.savefig('Plot6_SeqLength.eps', format='eps', dpi=600)
plt.close()

# Additional exercises part 1
print("\n--- Additional Exercises ---")
units_to_test = [16, 64]
for u in units_to_test:
    print(f"\nTraining LSTM with {u} units...")
    model = build_model('LSTM', 128, units=u)
    
    mean = np.mean(X_train, axis=(0, 1))
    std = np.std(X_train, axis=(0, 1))
    X_tr = (X_train - mean) / std
    X_ts = (X_test_sub - mean) / std
    
    start = time.time()
    model.fit(X_tr, y_train, batch_size=32, epochs=15, verbose=0)
    tt = time.time() - start
    
    y_pred = np.argmax(model.predict(X_ts, verbose=0), axis=-1)
    acc = np.mean(y_pred == y_test_sub)
    f1 = f1_score(y_test_sub, y_pred, average='macro', zero_division=0)
    print(f"LSTM ({u} units): Acc: {acc:.4f}, F1: {f1:.4f}, Params: {model.count_params()}, Time: {tt:.2f}s")
    
# 3. Add second recurrent layer
print("\nAdding a second LSTM layer...")
model2 = Sequential()
model2.add(Input(shape=(128, 9)))
model2.add(LSTM(32, return_sequences=True))
model2.add(LSTM(32))
model2.add(Dropout(0.2))
model2.add(Dense(16, activation='relu'))
model2.add(Dense(6, activation='softmax'))
model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
start = time.time()
model2.fit(X_tr, y_train, batch_size=32, epochs=15, verbose=0)
tt = time.time() - start
y_pred = np.argmax(model2.predict(X_ts, verbose=0), axis=-1)
f1 = f1_score(y_test_sub, y_pred, average='macro', zero_division=0)
print(f"2-layer LSTM: F1: {f1:.4f}, Params: {model2.count_params()}, Time: {tt:.2f}s")

# 4. Bidirectional LSTM
from tensorflow.keras.layers import Bidirectional
print("\nBidirectional LSTM...")
model_bi = Sequential()
model_bi.add(Input(shape=(128, 9)))
model_bi.add(Bidirectional(LSTM(32)))
model_bi.add(Dropout(0.2))
model_bi.add(Dense(16, activation='relu'))
model_bi.add(Dense(6, activation='softmax'))
model_bi.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
start = time.time()
model_bi.fit(X_tr, y_train, batch_size=32, epochs=15, verbose=0)
tt = time.time() - start
y_pred = np.argmax(model_bi.predict(X_ts, verbose=0), axis=-1)
f1 = f1_score(y_test_sub, y_pred, average='macro', zero_division=0)
print(f"Bidirectional LSTM: F1: {f1:.4f}, Params: {model_bi.count_params()}, Time: {tt:.2f}s")




In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, GRU, Dense
import cv2
import glob
import os

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})

print("Loading MobileNetV2...")
mobilenet = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', pooling='avg')

classes = ['Basketball', 'Archery', 'BandMarching']
num_videos_per_class = 5
frames_per_video = 10
H, W = 224, 224

videos = []
y = []

print("Extracting frames from real UCF101...")
for class_idx, c in enumerate(classes):
    path = f"UCF101_subset/train/{c}/*.avi"
    video_files = glob.glob(path)[:num_videos_per_class]
    for vf in video_files:
        cap = cv2.VideoCapture(vf)
        frames = []
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames < frames_per_video:
            continue
        indices = np.linspace(0, total_frames - 1, frames_per_video, dtype=int)
        for idx in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if idx in indices:
                frame = cv2.resize(frame, (W, H))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
        cap.release()
        
        while len(frames) < frames_per_video:
            frames.append(frames[-1])
        if len(frames) == frames_per_video:
            videos.append(np.array(frames))
            y.append(class_idx)

videos = np.array(videos).astype(np.float32)
y = np.array(y)
print(f"Extracted {len(videos)} videos.")

print("Extracting CNN features...")
features = []
for i in range(len(videos)):
    v = tf.keras.applications.mobilenet_v2.preprocess_input(videos[i])
    feat = mobilenet.predict(v, verbose=0)
    features.append(feat)

features = np.array(features)
print(f"CNN feature dimension: {features.shape[-1]}")
print(f"Resulting tensor shape supplied to recurrent network: {features.shape}")

idx = np.arange(len(features))
np.random.shuffle(idx)
split = int(0.7 * len(features))
X_train = features[idx[:split]]
y_train = y[idx[:split]]
X_val = features[idx[split:]]
y_val = y[idx[split:]]

print("Training CNN-LSTM...")
model = Sequential()
model.add(Input(shape=(10, 1280)))
model.add(LSTM(32))
model.add(Dense(3, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_lstm = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, verbose=0)

print("Training CNN-GRU...")
model_gru = Sequential()
model_gru.add(Input(shape=(10, 1280)))
model_gru.add(GRU(32))
model_gru.add(Dense(3, activation='softmax'))
model_gru.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_gru = model_gru.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, verbose=0)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['accuracy'], label='LSTM Train')
plt.plot(history_lstm.history['val_accuracy'], label='LSTM Val')
plt.plot(history_gru.history['accuracy'], label='GRU Train')
plt.plot(history_gru.history['val_accuracy'], label='GRU Val')
plt.title('Video Training & Val Acc', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Accuracy', fontweight='bold')
plt.legend(prop={'weight':'bold'})

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['loss'], label='LSTM Train')
plt.plot(history_lstm.history['val_loss'], label='LSTM Val')
plt.plot(history_gru.history['loss'], label='GRU Train')
plt.plot(history_gru.history['val_loss'], label='GRU Val')
plt.title('Video Training & Val Loss', fontweight='bold')
plt.xlabel('Epoch', fontweight='bold')
plt.ylabel('Loss', fontweight='bold')
plt.legend(prop={'weight':'bold'})
plt.tight_layout()
plt.savefig('Plot8_Video_Curves.eps', format='eps', dpi=600)
plt.close()

y_pred = np.argmax(model.predict(X_val, verbose=0), axis=-1)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 4))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Plot 9: Video Confusion Matrix (LSTM)', fontweight='bold')
plt.colorbar()
plt.ylabel('True', fontweight='bold')
plt.xlabel('Pred', fontweight='bold')
plt.savefig('Plot9_Video_CM.eps', format='eps', dpi=600)
plt.close()

# Plot 7: Video Sample Frames
plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(videos[0][i].astype(np.uint8))
    plt.axis('off')
plt.suptitle('Plot 7: Video Sample Frames (10 Frames)', fontweight='bold')
plt.tight_layout()
plt.savefig('Plot7_Video_Sample_Frames.eps', format='eps', dpi=600)
plt.close()

print("\n--- Seq2Seq ---")
def generate_seq2seq_data(num_samples=2000, seq_len=4, vocab_size=10):
    X = np.random.randint(1, vocab_size, size=(num_samples, seq_len))
    y = np.fliplr(X)
    return X, y

X_s, y_s = generate_seq2seq_data()
X_s_train, y_s_train = X_s[:1500], y_s[:1500]
X_s_val, y_s_val = X_s[1500:], y_s[1500:]

X_s_train_oh = tf.one_hot(X_s_train, 10)
y_s_train_oh = tf.one_hot(y_s_train, 10)
X_s_val_oh = tf.one_hot(X_s_val, 10)
y_s_val_oh = tf.one_hot(y_s_val, 10)

encoder_inputs = Input(shape=(None, 10))
encoder = LSTM(32, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None, 10))
decoder_lstm = LSTM(32, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(10, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

model_s2s = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model_s2s.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

dec_input_train = np.pad(y_s_train_oh[:, :-1, :], ((0,0), (1,0), (0,0)), mode='constant')
dec_input_val = np.pad(y_s_val_oh[:, :-1, :], ((0,0), (1,0), (0,0)), mode='constant')

print("Training Seq2Seq...")
hist_s2s = model_s2s.fit([X_s_train_oh, dec_input_train], y_s_train_oh, validation_data=([X_s_val_oh, dec_input_val], y_s_val_oh), batch_size=32, epochs=20, verbose=0)

print(f"Seq2Seq Training loss: {hist_s2s.history['loss'][-1]:.4f}")
print(f"Seq2Seq Validation loss: {hist_s2s.history['val_loss'][-1]:.4f}")

encoder_model = tf.keras.Model(encoder_inputs, encoder_states)
dec_state_input_h = Input(shape=(32,))
dec_state_input_c = Input(shape=(32,))
dec_states_inputs = [dec_state_input_h, dec_state_input_c]
dec_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=dec_states_inputs)
dec_states = [state_h, state_c]
dec_outputs = decoder_dense(dec_outputs)
decoder_model = tf.keras.Model([decoder_inputs] + dec_states_inputs, [dec_outputs] + dec_states)

def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1, 10))
    target_seq[0, 0, 0] = 1.0
    decoded_sentence = []
    for _ in range(4):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        decoded_sentence.append(sampled_token_index)
        target_seq = np.zeros((1, 1, 10))
        target_seq[0, 0, sampled_token_index] = 1.0
        states_value = [h, c]
    return decoded_sentence

print("\n--- Seq2Seq Examples ---")
print("Sample | Input Sequence | Predicted Output")
token_correct = 0
seq_correct = 0
total_tokens = 0
total_seqs = 0

for i in range(10):
    input_seq = X_s_val_oh[i:i+1]
    pred = decode_sequence(input_seq)
    inp_list = X_s_val[i].tolist()
    gt_list = y_s_val[i].tolist()
    if i < 5:
        print(f"{i+1} | {inp_list} | {pred}")
    
    t_corr = sum([1 for p, g in zip(pred, gt_list) if p == g])
    token_correct += t_corr
    total_tokens += 4
    if t_corr == 4:
        seq_correct += 1
    total_seqs += 1
    
print(f"Token Accuracy: {token_correct/total_tokens:.4f}")
print(f"Sequence Accuracy: {seq_correct/total_seqs:.4f}")

# Additional 7
print("\nSeq2Seq with different lengths (len 4 -> len 3)")
y_diff = y_s[:, :-1]
y_diff_train = y_diff[:1500]
y_diff_val = y_diff[1500:]
y_diff_train_oh = tf.one_hot(y_diff_train, 10)
y_diff_val_oh = tf.one_hot(y_diff_val, 10)
dec_input_train_diff = np.pad(y_diff_train_oh[:, :-1, :], ((0,0), (1,0), (0,0)), mode='constant')
dec_input_val_diff = np.pad(y_diff_val_oh[:, :-1, :], ((0,0), (1,0), (0,0)), mode='constant')

decoder_inputs_diff = Input(shape=(None, 10))
decoder_lstm_diff = LSTM(32, return_sequences=True, return_state=True)
decoder_outputs_diff, _, _ = decoder_lstm_diff(decoder_inputs_diff, initial_state=encoder_states)
decoder_dense_diff = Dense(10, activation='softmax')
decoder_outputs_diff = decoder_dense_diff(decoder_outputs_diff)

model_s2s_diff = tf.keras.Model([encoder_inputs, decoder_inputs_diff], decoder_outputs_diff)
model_s2s_diff.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
hist_diff = model_s2s_diff.fit([X_s_train_oh, dec_input_train_diff], y_diff_train_oh, epochs=5, verbose=0)
print(f"Diff length seq2seq loss: {hist_diff.history['loss'][-1]:.4f}")

